# 02 · EDA en cleaning

**Auteurs:** _vul namen en concrete bijdragen in vóór indiening_  
**Doel:** Datakwaliteit onderzoeken en een inhoudelijk verhaal met aggregaties en grafieken bouwen.

Elke codecel wordt voorafgegaan door uitleg. Voer de notebooks in nummervolgorde uit
vanaf de repository-root. Resultaten moeten door het team zelf worden gecontroleerd en
geïnterpreteerd; synthetische CI-data gelden nooit als onderzoeksbewijs.

In [ ]:
# Maak lokale projectimports betrouwbaar in VS Code, JupyterLab en nbconvert.
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "pyproject.toml").exists() and (path / "src").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Projectroot niet gevonden; open de map Brent als VS Code-workspace.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Projectroot: {PROJECT_ROOT}")

## Laden

Notebook 03 maakt het Parquet-bestand. Voor de EDA nemen we maximaal één miljoen gelijkmatig verdeelde rijen om plots responsief te houden.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import PROCESSED_DIR

sns.set_theme(style="whitegrid")
path = PROCESSED_DIR / "trips.parquet"
trips = pd.read_parquet(path).sort_values("started_at")
if len(trips) > 1_000_000:
    trips = trips.iloc[np.linspace(0, len(trips)-1, 1_000_000, dtype=int)].copy()
print(f"EDA-steekproef: {len(trips):,} ritten, {trips.started_at.min()} t/m {trips.started_at.max()}")
display(trips.head())

## Datakwaliteit

We kwantificeren ontbrekende waarden en duplicaten. Missende eindstations mogen voor ons voorspeldoel blijven bestaan; een ontbrekend startstation niet.

In [ ]:
quality = pd.DataFrame({
    "dtype": trips.dtypes.astype(str),
    "missing_n": trips.isna().sum(),
    "missing_pct": trips.isna().mean().mul(100).round(2),
    "unique": trips.nunique(dropna=False),
}).sort_values("missing_pct", ascending=False)
display(quality)
print("Dubbele ride_id's:", trips.ride_id.duplicated().sum())
display(trips.duration_minutes.describe(percentiles=[.5, .9, .95, .99]).to_frame())

## Cleaning-impact

De audit telt redenen niet exclusief: één rij kan tegelijk een ongeldige duur én locatie hebben. Daarom tellen redenkolommen niet op tot het totaal verwijderd.

In [ ]:
audit_path = PROCESSED_DIR / "cleaning_audit.json"
if audit_path.exists():
    audit = json.loads(audit_path.read_text())
    summary = {k: v for k, v in audit.items() if k != "chunks"}
    display(pd.Series(summary, name="waarde"))
    print(f"Behoud: {audit['output_rows'] / audit['input_rows']:.1%}")

## Aggregatie 1 — wanneer wordt gereden?

Aantal ritten en mediane duur beantwoorden verschillende vragen: gebruiksvolume versus typisch gedrag.

In [ ]:
trips["hour"] = trips.started_at.dt.hour
hourly = trips.groupby(["hour", "member_casual"], observed=True).agg(
    rides=("ride_id", "size"), median_minutes=("duration_minutes", "median")
).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.lineplot(data=hourly, x="hour", y="rides", hue="member_casual", ax=axes[0])
sns.lineplot(data=hourly, x="hour", y="median_minutes", hue="member_casual", ax=axes[1])
axes[0].set(title="Ritten per startuur", ylabel="ritten in steekproef")
axes[1].set(title="Mediane ritduur per startuur", ylabel="minuten")
plt.tight_layout()
display(hourly.head())

## Aggregatie 2 — weekritme

Weekdag wordt geordend, zodat alfabetische sortering geen misleidend patroon maakt.

In [ ]:
order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
trips["weekday"] = pd.Categorical(trips.started_at.dt.day_name().str[:3], order, ordered=True)
daily = trips.groupby(["weekday", "member_casual"], observed=True).agg(
    rides=("ride_id", "size"), median_minutes=("duration_minutes", "median")
).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=daily, x="weekday", y="rides", hue="member_casual", ax=axes[0])
sns.pointplot(data=daily, x="weekday", y="median_minutes", hue="member_casual", ax=axes[1])
axes[0].set(title="Ritvolume per weekdag", ylabel="ritten")
axes[1].set(title="Typische ritduur per weekdag", ylabel="mediane minuten")
plt.tight_layout()

## Aggregatie 3 — stations

Alleen stations met minstens 500 ritten worden voor mediane duur gerangschikt; kleine groepen leveren instabiele extremen.

In [ ]:
stations = trips.groupby(["start_station_id", "start_station_name"], observed=True).agg(
    rides=("ride_id", "size"), median_minutes=("duration_minutes", "median"),
    casual_share=("member_casual", lambda s: (s == "casual").mean()),
).reset_index()
stable = stations.query("rides >= 500").nlargest(15, "rides")
plt.figure(figsize=(9, 6))
sns.barplot(data=stable, y="start_station_name", x="rides", color="#4f8f67")
plt.title("Top-15 startstations in de EDA-steekproef")
plt.tight_layout()
display(stations.sort_values("rides", ascending=False).head(20))

## Verdeling en outliers

De rechterstaart motiveert MAE en een log-getransformeerde target bij modelleren. De grens van 180 minuten is een vooraf gedocumenteerde plausibiliteitsregel, geen op de testscore afgestemde keuze.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(trips.duration_minutes, bins=80, ax=axes[0])
sns.boxplot(data=trips, x="member_casual", y="duration_minutes", showfliers=False, ax=axes[1])
axes[0].set(xlabel="duur (minuten)", title="Ritduur na plausibiliteitsfilter")
axes[1].set(xlabel="type rijder", ylabel="duur (minuten)", title="Verdeling zonder getekende uitschieters")
plt.tight_layout()

## Interpretatiechecklist

Vul na uitvoering concrete getallen in: periode en N, grootste piekuren, member/casual-verschil, missings, cleaning-impact en beperkingen. Formuleer associaties; deze observationele data ondersteunen geen causale claims.